# Task 1: Cheater Detection (Binary Classification — Binary $F_2$)

**Objective**: Predict whether a player account is a cheater (`is_cheater` $\in \{0, 1\}$), evaluated by **Binary $F_2$ score** (weights recall 4× higher than precision).

### Pipeline Summary
1. **Essential EDA** — Identify 16 pure-noise features, confirm missingness is MCAR (Missing Completely At Random), and verify the exact algebraic link between `level`, `account_age_days`, and `level_progression_speed`.
2. **Feature Engineering** — Drop noise features, compute algebraic missing-value bounds and Bayesian zone posteriors, and add domain ratios/interactions.
3. **3-Component Diverse Ensemble**:
   - **Model A (weight 0.70)**: Multi-seed LightGBM trained with **MCAR missing-value masking augmentation**.
   - **Model B (weight 0.20)**: 4-family **Logistic Regression stack** (LightGBM, ExtraTrees, PyTorch MLP, Spline Linear SVM) for model diversity.
   - **Model C (weight 0.10)**: Tuned **XGBoost + LightGBM + CatBoost** blend with categorical binning.
4. **Leak-Free Post-Processing** — Align component scores using each model's training OOF empirical CDF (ECDF) so no test-set statistics are used, then apply a fixed OOF $F_2$-optimal threshold.

In [1]:
import os
import glob
import time
import warnings
import numpy as np
import pandas as pd

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import torch
import torch.nn as nn

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, fbeta_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, QuantileTransformer, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import ExtraTreesClassifier

warnings.filterwarnings("ignore")

def find_data_dir(subdir: str) -> str:
    for p in [os.path.join("..", "cpe342-kaggle-main", subdir), os.path.join("cpe342-kaggle-main", subdir)]:
        if os.path.isdir(p):
            return os.path.abspath(p)
    hits = glob.glob(f"/kaggle/input/**/{subdir}/train.csv", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError(f"Dataset not found for {subdir}. Expected in cpe342-kaggle-main/{subdir}.")

DATA_DIR = find_data_dir("01_binary_classification")
OUT_CSV = "task1_predictions.csv"
TARGET = "is_cheater"
N_JOBS = os.cpu_count() or 4
torch.set_num_threads(min(2, N_JOBS))

tr_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
te_raw = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

lab_mask = tr_raw[TARGET].notna().values
lab = tr_raw[lab_mask].reset_index(drop=True)
unl = tr_raw[~lab_mask].reset_index(drop=True)
ref_train = pd.concat([lab, unl], ignore_index=True)  # all training rows for label-free stats

y = lab[TARGET].astype(int).values
test_ids = np.array([f"ANS{i+1:05d}" for i in range(len(te_raw))])

print(f"Train total: {len(tr_raw):,} | Labeled: {len(lab):,} | Unlabeled: {len(unl):,} | Test: {len(te_raw):,}")
print(f"Positive (cheater) rate in labeled train: {y.mean():.4f}")

Train total: 104,000 | Labeled: 90,816 | Unlabeled: 13,184 | Test: 25,889
Positive (cheater) rate in labeled train: 0.3493


## 1. Essential EDA & Design Decisions

We run three quick checks on the training data to justify our preprocessing and modeling decisions:
1. **Univariate Signal Check**: Compute single-feature ROC-AUC on labeled training rows. Features with AUC $\approx 0.50$ carry no predictive signal $\rightarrow$ **Decision: drop 16 noise features, keep 15 signal features.**
2. **Missingness Mechanism (MCAR)**: Compare the cheater rate when a feature is missing vs. observed. Nearly identical rates confirm missingness is MCAR $\rightarrow$ **Decision: apply MCAR random-masking augmentation during training to improve robustness to missing values.**
3. **Algebraic Identity Check**: Inspect $r = \frac{\text{level\_progression\_speed} \times \text{account\_age\_days}}{\text{level}}$. Values lie tightly in $[0.15, 1.08]$ $\rightarrow$ **Decision: use this identity to compute exact missing-value bounds and Bayesian zone posteriors whenever `account_age_days` or `level` is missing.**

In [2]:
# 1. Univariate ROC-AUC check across all raw features
raw_features = [c for c in lab.columns if c not in ("player_id", TARGET)]
aucs = {}
for c in raw_features:
    ok = lab[c].notna().values
    a = roc_auc_score(y[ok], lab.loc[ok, c])
    aucs[c] = max(a, 1.0 - a)

auc_s = pd.Series(aucs).sort_values(ascending=False)
SIGNAL = [
    "kill_death_ratio", "headshot_percentage", "win_rate", "accuracy_score", "kill_consistency",
    "reaction_time_ms", "account_age_days", "level", "level_progression_speed", "friend_network_size",
    "reports_received", "aiming_smoothness", "spray_control_score", "game_sense_score", "crosshair_placement"
]
noise_cols = [c for c in raw_features if c not in SIGNAL]
print(f"Signal features ({len(SIGNAL)}): AUC range [{auc_s[SIGNAL].min():.4f}, {auc_s[SIGNAL].max():.4f}]")
print(f"Noise features  ({len(noise_cols)}): AUC range [{auc_s[noise_cols].min():.4f}, {auc_s[noise_cols].max():.4f}] -> DROPPED")

# 2. MCAR check on top features with missing values
print("\nMCAR check (cheater rate when NaN vs observed):")
for c in ["reports_received", "crosshair_placement", "account_age_days", "level", "friend_network_size"]:
    m = lab[c].isna().values
    print(f"  {c:22s} | missing={m.mean()*100:5.2f}% | P(cheat|NaN)={y[m].mean():.4f} vs P(cheat|obs)={y[~m].mean():.4f}")

# 3. Algebraic identity check: r = level_progression_speed * account_age_days / level
r_obs = (lab["level_progression_speed"] * lab["account_age_days"] / lab["level"]).dropna()
print(f"\nIdentity check r = lps * age / level: min={r_obs.min():.4f}, p01={r_obs.quantile(0.01):.4f}, "
      f"p99={r_obs.quantile(0.99):.4f}, max={r_obs.max():.4f}")

Signal features (15): AUC range [0.5106, 0.6965]
Noise features  (16): AUC range [0.5002, 0.5033] -> DROPPED

MCAR check (cheater rate when NaN vs observed):
  reports_received       | missing=22.29% | P(cheat|NaN)=0.3476 vs P(cheat|obs)=0.3498
  crosshair_placement    | missing=12.81% | P(cheat|NaN)=0.3508 vs P(cheat|obs)=0.3491
  account_age_days       | missing=14.47% | P(cheat|NaN)=0.3469 vs P(cheat|obs)=0.3497
  level                  | missing= 7.17% | P(cheat|NaN)=0.3490 vs P(cheat|obs)=0.3494
  friend_network_size    | missing=22.29% | P(cheat|NaN)=0.3478 vs P(cheat|obs)=0.3498

Identity check r = lps * age / level: min=0.1500, p01=0.1593, p99=1.0712, max=1.0800


## 2. Feature Engineering (Bounds, Zone Flags & Bayesian Posteriors)

All medians, standard deviations, frequency tables, and marginal priors are fitted strictly on training data (`ref_train`) to prevent data leakage.
- **Algebraic bounds**: When `account_age_days` is missing, `level` and `level_progression_speed` bound it to $[0.15\,\text{level}/\text{lps},\; 1.08\,\text{level}/\text{lps}]$ (and analogously for missing `level`).
- **Bayesian zone posteriors**: Using the empirical training prior over `account_age_days` and `level` combined with the uniform likelihood $r \sim U(0.15, 1.08)$, we integrate the posterior probability that a missing value falls inside each class-exclusive range.

In [3]:
AGE_ZONES = {
    "cheat_110_120": (110, 120),
    "legit_lt15": (0, 15),
    "legit_180_200": (180, 200),
    "legit_ge1000": (1000, 1e9),
    "young_lt110": (0, 110),
    "mid_120_180": (120, 180),
}
LVL_ZONES = {"cheat_lt28": (0, 28), "legit_gt96": (96, 1e9)}


def build_core_features(df: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    X = df[SIGNAL].copy()
    lv, ag, lp = df["level"], df["account_age_days"], df["level_progression_speed"]
    for c in ["reports_received", "friend_network_size", "account_age_days", "crosshair_placement", "level"]:
        X[f"na_{c}"] = df[c].isna().astype(np.int8)
    X["n_missing"] = df[SIGNAL].isna().sum(axis=1)

    # Algebraic bounds & geometric-mean fill for missing age / level
    X["age_lo"] = lv * 0.15 / lp
    X["age_hi"] = lv * 1.08 / lp
    X["age_filled"] = ag.fillna(np.sqrt(X["age_lo"] * X["age_hi"]))
    X["lv_lo"] = lp * ag / 1.08
    X["lv_hi"] = lp * ag / 0.15
    X["lvl_filled"] = lv.fillna(np.sqrt(X["lv_lo"] * X["lv_hi"]))
    X["r_hat"] = lp * ag / lv

    # Class-exclusive behavioral zone flags
    rep, ch, age, fr = df["reports_received"], df["crosshair_placement"], X["age_filled"], df["friend_network_size"]
    X["zone_cheat"] = ((rep == 4) | (ch < 0.82) | (X["lvl_filled"] < 28)).astype(np.int8)
    X["zone_legit"] = (
        (rep >= 20) | (ch > 0.94) | (X["lvl_filled"] > 96) |
        (age > 1000) | (age < 15) | (fr < 10) | (fr > 269)
    ).astype(np.int8)

    for c in ["account_age_days", "friend_network_size", "level_progression_speed", "reports_received"]:
        X[f"log_{c}"] = np.log1p(df[c].clip(lower=0))

    X["kd_x_hs"] = df["kill_death_ratio"] * df["headshot_percentage"]
    X["acc_x_cross"] = df["accuracy_score"] * df["crosshair_placement"]
    X["reports_per_100d"] = 100 * rep / (age + 1)
    X["friends_per_100d"] = 100 * fr / (age + 1)
    X["level_per_100d"] = 100 * X["lvl_filled"] / (age + 1)

    sk = [
        "kill_death_ratio", "headshot_percentage", "accuracy_score", "win_rate", "kill_consistency",
        "aiming_smoothness", "spray_control_score", "game_sense_score", "crosshair_placement"
    ]
    mu, sd = ref[sk].median(), ref[sk].std() + 1e-9
    Z = (df[sk] - mu) / sd
    X["skill_z_sum"] = Z.sum(axis=1, min_count=3)
    X["skill_z_std"] = Z.std(axis=1)

    X["freq_reports"] = rep.map(ref["reports_received"].value_counts(normalize=True)).astype(float)
    X["freq_friends"] = fr.map(ref["friend_network_size"].value_counts(normalize=True)).astype(float)
    X["age_segment"] = pd.cut(age, [-np.inf, 15, 100, 200, 1000, np.inf], labels=False).astype(float)
    return X.replace([np.inf, -np.inf], np.nan)


def _normalize_posterior(obs_prior, lik):
    w = lik * obs_prior[None, :]
    s = w.sum(1, keepdims=True)
    return np.where(s > 0, w / np.where(s > 0, s, 1), np.nan)


def build_zone_posteriors(df: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    ag, lv, lp = df["account_age_days"].values, df["level"].values, df["level_progression_speed"].values

    # Posterior over missing account_age_days
    gA = np.arange(0.5, 1500, 1.0)
    pA = np.histogram(ref["account_age_days"].dropna(), np.arange(0, 1501, 1.0))[0].astype(float) + 0.5
    m = np.isnan(ag) & ~np.isnan(lv) & ~np.isnan(lp)
    E_age = np.full(len(df), np.nan)
    for nm in AGE_ZONES:
        out[f"pa_{nm}"] = np.nan
    if m.any():
        r = lp[m, None] * gA[None, :] / lv[m, None]
        lik = ((r >= 0.15) & (r <= 1.08)) * gA[None, :]
        P = _normalize_posterior(pA, lik)
        E_age[m] = (P * gA).sum(1)
        for nm, (a, b) in AGE_ZONES.items():
            out.loc[m, f"pa_{nm}"] = P[:, (gA >= a) & (gA < b)].sum(1)
    out["age_post_mean"] = np.where(np.isnan(ag), E_age, ag)
    o = ~np.isnan(ag)
    for nm, (a, b) in AGE_ZONES.items():
        out.loc[o, f"pa_{nm}"] = ((ag[o] >= a) & (ag[o] < b)).astype(float)

    # Posterior over missing level
    gL = np.arange(25.05, 100, 0.1)
    pL = np.histogram(ref["level"].dropna(), np.arange(25.0, 100.01, 0.1))[0].astype(float)[:len(gL)] + 0.5
    m = np.isnan(lv) & ~np.isnan(ag) & ~np.isnan(lp)
    E_l = np.full(len(df), np.nan)
    for nm in LVL_ZONES:
        out[f"pl_{nm}"] = np.nan
    if m.any():
        r = lp[m, None] * ag[m, None] / gL[None, :]
        lik = ((r >= 0.15) & (r <= 1.08)) / gL[None, :]
        P = _normalize_posterior(pL, lik)
        E_l[m] = (P * gL).sum(1)
        for nm, (a, b) in LVL_ZONES.items():
            out.loc[m, f"pl_{nm}"] = P[:, (gL >= a) & (gL < b)].sum(1)
    out["lvl_post_mean"] = np.where(np.isnan(lv), E_l, lv)
    o = ~np.isnan(lv)
    for nm, (a, b) in LVL_ZONES.items():
        out.loc[o, f"pl_{nm}"] = ((lv[o] >= a) & (lv[o] < b)).astype(float)

    out["p_cheat_zone"] = 1 - (1 - out["pa_cheat_110_120"].fillna(0)) * (1 - out["pl_cheat_lt28"].fillna(0))
    out["p_legit_zone"] = 1 - (
        1 - out[["pa_legit_lt15", "pa_legit_180_200", "pa_legit_ge1000"]].fillna(0).sum(1).clip(0, 1)
    ) * (1 - out["pl_legit_gt96"].fillna(0))
    return out


def build_features_full(df: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    return pd.concat([build_core_features(df, ref), build_zone_posteriors(df, ref)], axis=1)


X_full = build_features_full(lab, ref_train)
Xt_full = build_features_full(te_raw, ref_train)
print(f"Engineered feature matrix: train={X_full.shape}, test={Xt_full.shape}")

Engineered feature matrix: train=(90816, 56), test=(25889, 56)


## 3. Model A (Weight 0.70): LightGBM with MCAR Masking Augmentation

- **Design**: 3 random seeds (`42, 7, 2024`) × 5 stratified folds × 3 bags (`learning_rate=0.02`, early stopping on a 10% inner holdout).
- **Augmentation**: In each training fold, 4 masked copies are generated by randomly hiding each signal feature at its observed missing rate and recomputing all engineered features. Validation and test rows are never masked.

In [4]:
LGB_PARAMS = dict(
    num_leaves=39, min_child_samples=8, subsample=0.814, subsample_freq=1,
    colsample_bytree=0.455, reg_lambda=4.01, reg_alpha=0.0045,
    min_split_gain=0.02, max_bin=511, verbose=-1, n_jobs=N_JOBS,
)

SEEDS = (42, 7, 2024)
N_BAG, N_COPIES, MASK_RATE, LR = 3, 4, 1.0, 0.02
miss_rate = lab[SIGNAL].isna().mean().values


def make_masked_copy(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    d = df.copy()
    M = rng.random((len(d), len(SIGNAL))) < miss_rate[None, :] * MASK_RATE
    v = d[SIGNAL].values.copy()
    v[M] = np.nan
    d[SIGNAL] = v
    return build_features_full(d, ref_train)


t0 = time.time()
oofs_a, tests_a = [], []
for s in SEEDS:
    oof_s = np.zeros(len(y))
    tp_s = np.zeros(len(Xt_full))
    folds = list(StratifiedKFold(5, shuffle=True, random_state=s).split(X_full, y))
    for i in range(N_BAG):
        for k, (a, b) in enumerate(folds):
            rng = np.random.default_rng(s + k + 100 * i)
            a_perm = rng.permutation(a)
            cut = int(0.9 * len(a_perm))
            tr_i, va_i = a_perm[:cut], a_perm[cut:]
            Xa = pd.concat(
                [X_full.iloc[tr_i]] + [make_masked_copy(lab.iloc[tr_i], rng) for _ in range(N_COPIES)],
                ignore_index=True,
            )
            m = lgb.LGBMClassifier(n_estimators=20000, learning_rate=LR, random_state=s + k + 1000 * i, **LGB_PARAMS)
            m.fit(
                Xa, np.tile(y[tr_i], N_COPIES + 1),
                eval_set=[(X_full.iloc[va_i], y[va_i])],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(200, verbose=False)],
            )
            oof_s[b] += m.predict_proba(X_full.iloc[b])[:, 1] / N_BAG
            tp_s += m.predict_proba(Xt_full)[:, 1] / (5 * N_BAG)
    print(f"  [Model A] seed {s} OOF AUC: {roc_auc_score(y, oof_s):.5f} ({(time.time()-t0)/60:.1f} min)")
    oofs_a.append(oof_s)
    tests_a.append(tp_s)

c1_oof = np.mean(oofs_a, axis=0)
c1_test = np.mean(tests_a, axis=0)
print(f"[Model A: MCAR-Augmented LightGBM] 3-seed OOF AUC: {roc_auc_score(y, c1_oof):.5f}")

  [Model A] seed 42 OOF AUC: 0.89446 (20.1 min)


  [Model A] seed 7 OOF AUC: 0.89455 (35.8 min)


  [Model A] seed 2024 OOF AUC: 0.89457 (50.6 min)
[Model A: MCAR-Augmented LightGBM] 3-seed OOF AUC: 0.89482


## 4. Model B (Weight 0.20): 4-Family Diversity Stack

To reduce single-family variance, we stack 4 distinct model families using a 5-fold Logistic Regression meta-learner over out-of-fold logits:
1. LightGBM (gradient boosted trees)
2. ExtraTrees (randomized tree ensemble)
3. PyTorch MLP (2-layer neural network on quantile-normalized inputs)
4. Spline + Linear SVM (smooth nonlinear linear model)

In [5]:
SIG_STACK = [
    "reports_received", "crosshair_placement", "kill_death_ratio", "headshot_percentage",
    "game_sense_score", "accuracy_score", "level", "spray_control_score", "aiming_smoothness",
    "win_rate", "account_age_days", "friend_network_size", "kill_consistency",
    "reaction_time_ms", "level_progression_speed"
]


def build_stack_tree_features(df: pd.DataFrame) -> pd.DataFrame:
    X = df[SIG_STACK].copy()
    a, l, s = df.account_age_days, df.level, df.level_progression_speed
    X["age_lo"] = (l * 0.15 / s).where(a.isna(), a)
    X["age_hi"] = (l * 1.08 / s).where(a.isna(), a)
    X["lv_lo"] = (s * a / 1.08).where(l.isna(), l)
    X["lv_hi"] = (s * a / 0.15).where(l.isna(), l)
    X["r"] = s * a / l
    X["fr_age"] = df.friend_network_size / a
    X["rep_age"] = df.reports_received / a
    return X


def build_stack_smooth_features(df: pd.DataFrame) -> np.ndarray:
    X = build_stack_tree_features(df)
    for c in ["level_progression_speed", "fr_age", "rep_age", "age_lo", "age_hi", "lv_lo", "lv_hi", "r"]:
        X[c] = np.log1p(X[c].clip(lower=0))
    X["age_hi"] = X["age_hi"].clip(upper=8)
    X["age_lo"] = X["age_lo"].clip(upper=8)
    M = df[SIG_STACK].isna().astype(float).add_suffix("_na")
    return pd.concat([X, M], axis=1).values


X_st_gbm, Xt_st_gbm = build_stack_tree_features(lab).values, build_stack_tree_features(te_raw).values
X_st_nt, Xt_st_nt = build_stack_smooth_features(lab), build_stack_smooth_features(te_raw)
folds_42 = list(StratifiedKFold(5, shuffle=True, random_state=42).split(lab, y))


def run_cv_base(fit_pred, Xtr, Xte):
    oof = np.zeros(len(y))
    for a, b in folds_42:
        oof[b] = fit_pred(Xtr[a], y[a], [Xtr[b]])[0]
    pt = fit_pred(Xtr, y, [Xte])[0]
    return oof, pt


def fp_gbm(Xa, ya, Bs):
    m = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.03, num_leaves=31, min_child_samples=50,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=5,
        n_jobs=N_JOBS, verbose=-1, random_state=42
    ).fit(Xa, ya)
    return [m.predict_proba(x)[:, 1] for x in Bs]


def fp_et(Xa, ya, Bs):
    m = make_pipeline(
        SimpleImputer(strategy="constant", fill_value=-1),
        ExtraTreesClassifier(200, min_samples_leaf=10, max_features=0.5, n_jobs=N_JOBS, random_state=0)
    ).fit(Xa, ya)
    return [m.predict_proba(x)[:, 1] for x in Bs]


def fp_mlp(Xa, ya, Bs):
    pp = make_pipeline(
        SimpleImputer(strategy="median"),
        QuantileTransformer(n_quantiles=200, output_distribution="normal", subsample=50000, random_state=0)
    ).fit(Xa)
    to_t = lambda x: torch.tensor(np.nan_to_num(pp.transform(x)), dtype=torch.float32)
    A_, y_ = to_t(Xa), torch.tensor(ya, dtype=torch.float32)
    torch.manual_seed(0)
    m = nn.Sequential(
        nn.Linear(A_.shape[1], 128), nn.ReLU(), nn.Dropout(0.1),
        nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1)
    )
    opt = torch.optim.AdamW(m.parameters(), 2e-3, weight_decay=1e-3)
    E = 12
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, 4e-3, total_steps=E * ((len(A_) + 511) // 512))
    for _ in range(E):
        m.train()
        perm = torch.randperm(len(A_))
        for i in range(0, len(A_), 512):
            j = perm[i:i + 512]
            opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(m(A_[j]).squeeze(1), y_[j]).backward()
            opt.step()
            sch.step()
    m.eval()
    with torch.no_grad():
        return [torch.sigmoid(m(to_t(x)).squeeze(1)).numpy() for x in Bs]


def fp_svm_spl(Xa, ya, Bs):
    m = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        SplineTransformer(n_knots=8, degree=3, extrapolation="constant"),
        LinearSVC(C=0.05, dual=False, max_iter=3000)
    ).fit(Xa, ya)
    return [1.0 / (1.0 + np.exp(-m.decision_function(x))) for x in Bs]


base_oofs, base_tests = [], []
for nm, fn, Xtr, Xte in [
    ("LightGBM", fp_gbm, X_st_gbm, Xt_st_gbm),
    ("ExtraTrees", fp_et, X_st_nt, Xt_st_nt),
    ("PyTorch MLP", fp_mlp, X_st_nt, Xt_st_nt),
    ("Spline LinearSVC", fp_svm_spl, X_st_nt, Xt_st_nt),
]:
    oo, tt = run_cv_base(fn, Xtr, Xte)
    print(f"  [Model B base] {nm:18s} OOF AUC: {roc_auc_score(y, oo):.5f}")
    base_oofs.append(oo)
    base_tests.append(tt)

logit = lambda p: np.log(np.clip(p, 1e-4, 1 - 1e-4) / (1 - np.clip(p, 1e-4, 1 - 1e-4)))
O_meta = np.column_stack([logit(p) for p in base_oofs])
T_meta = np.column_stack([logit(p) for p in base_tests])

c2_oof = np.zeros(len(y))
for a, b in folds_42:
    meta = LogisticRegression(C=1.0).fit(O_meta[a], y[a])
    c2_oof[b] = meta.predict_proba(O_meta[b])[:, 1]
meta_full = LogisticRegression(C=1.0).fit(O_meta, y)
c2_test = meta_full.predict_proba(T_meta)[:, 1]
print(f"[Model B: 4-Family Stack] OOF AUC: {roc_auc_score(y, c2_oof):.5f} | meta-weights: {np.round(meta_full.coef_[0], 3)}")

  [Model B base] LightGBM           OOF AUC: 0.89285


  [Model B base] ExtraTrees         OOF AUC: 0.86112


  [Model B base] PyTorch MLP        OOF AUC: 0.84681


  [Model B base] Spline LinearSVC   OOF AUC: 0.83907


[Model B: 4-Family Stack] OOF AUC: 0.89283 | meta-weights: [ 1.234 -0.002  0.063 -0.518]


## 5. Model C (Weight 0.10): Tuned 3-GBM Ensemble (XGBoost + LightGBM + CatBoost)

Trains 5-fold XGBoost, LightGBM, and CatBoost using tuned hyperparameters and explicit categorical quantile bins (`reports_cat`, `age_segment`, `friends_bin`, `level_bin`, `crosshair_bin`), combined with weights `0.1834 / 0.4848 / 0.3318`.

In [6]:
CAT_COLS = ["reports_cat", "age_segment", "friends_bin", "level_bin", "crosshair_bin"]

def build_binned_gbm_features(train_df: pd.DataFrame, test_df: pd.DataFrame):
    stats = {}
    sk = [
        "kill_death_ratio", "headshot_percentage", "accuracy_score", "win_rate",
        "kill_consistency", "aiming_smoothness", "spray_control_score",
        "game_sense_score", "crosshair_placement"
    ]
    stats["mu"] = train_df[sk].median()
    stats["sd"] = train_df[sk].std() + 1e-9
    for c, nb in [("friend_network_size", 24), ("level", 20), ("crosshair_placement", 20)]:
        stats[f"edges_{c}"] = np.unique(np.nanquantile(train_df[c], np.linspace(0, 1, nb + 1)))
    stats["freq_reports"] = train_df["reports_received"].value_counts(normalize=True).to_dict()
    stats["freq_friends"] = train_df["friend_network_size"].value_counts(normalize=True).to_dict()

    def transform(df):
        X = df[SIGNAL].copy()
        lv, ag, lp = df["level"], df["account_age_days"], df["level_progression_speed"]
        for c in ["reports_received", "friend_network_size", "account_age_days", "crosshair_placement", "level"]:
            X[f"na_{c}"] = df[c].isna().astype(np.int8)
        X["n_missing"] = df[SIGNAL].isna().sum(axis=1)
        X["age_lo"] = lv * 0.15 / lp
        X["age_hi"] = lv * 1.08 / lp
        X["age_filled"] = ag.fillna(np.sqrt(X["age_lo"] * X["age_hi"]))
        X["lv_lo"] = lp * ag / 1.08
        X["lv_hi"] = lp * ag / 0.15
        X["lvl_filled"] = lv.fillna(np.sqrt(X["lv_lo"] * X["lv_hi"]))
        X["r_hat"] = lp * ag / lv
        rep, ch, age, fr = df["reports_received"], df["crosshair_placement"], X["age_filled"], df["friend_network_size"]
        X["zone_cheat"] = ((rep == 4) | (ch < 0.82) | (X["lvl_filled"] < 28)).astype(np.int8)
        X["zone_legit"] = (
            (rep >= 20) | (ch > 0.94) | (X["lvl_filled"] > 96) |
            (age > 1000) | (age < 15) | (fr < 10) | (fr > 269)
        ).astype(np.int8)
        for c in ["account_age_days", "friend_network_size", "level_progression_speed", "reports_received"]:
            X[f"log_{c}"] = np.log1p(df[c].clip(lower=0))
        X["kd_x_hs"] = df["kill_death_ratio"] * df["headshot_percentage"]
        X["acc_x_cross"] = df["accuracy_score"] * df["crosshair_placement"]
        X["reports_per_100d"] = 100 * rep / (age + 1)
        X["friends_per_100d"] = 100 * fr / (age + 1)
        X["level_per_100d"] = 100 * X["lvl_filled"] / (age + 1)
        Z = (df[sk] - stats["mu"]) / stats["sd"]
        X["skill_z_sum"] = Z.sum(axis=1, min_count=3)
        X["skill_z_std"] = Z.std(axis=1)
        X["freq_reports"] = rep.map(stats["freq_reports"]).astype(float)
        X["freq_friends"] = fr.map(stats["freq_friends"]).astype(float)
        X["reports_cat"] = rep.fillna(-1).astype(int).astype(str)
        X["age_segment"] = pd.cut(age, [-np.inf, 15, 100, 200, 1000, np.inf],
                                  labels=["a<15", "a15-100", "a100-200", "a200-1000", "a>1000"]).astype(str)
        for c, nm in [("friend_network_size", "friends_bin"), ("level", "level_bin"), ("crosshair_placement", "crosshair_bin")]:
            X[nm] = pd.cut(df[c], stats[f"edges_{c}"], include_lowest=True, labels=False).fillna(-1).astype(int).astype(str)
        X = X.replace([np.inf, -np.inf], np.nan)
        for c in CAT_COLS:
            X[c] = X[c].astype("category")
        return X

    A, B = transform(train_df), transform(test_df)
    for c in CAT_COLS:
        sa, sb = A[c].astype(object).map(str), B[c].astype(object).map(str)
        cats = sorted(set(sa) | set(sb))
        A[c] = pd.Categorical(sa, categories=cats)
        B[c] = pd.Categorical(sb, categories=cats)
    return A, B


X_p, Xt_p = build_binned_gbm_features(lab, te_raw)

XGB_OPT = dict(
    max_depth=7, min_child_weight=1.4104607765610724, subsample=0.9983803458250254,
    colsample_bytree=0.6964613905152919, reg_lambda=15.435762030980067, reg_alpha=11.784650859403348,
    gamma=0.03686656101587725, max_bin=128, grow_policy="lossguide",
)
LGB_OPT = dict(
    num_leaves=39, min_child_samples=8, subsample=0.8140545534012371, subsample_freq=1,
    colsample_bytree=0.4548065558904867, reg_lambda=4.0107045889162665, reg_alpha=0.004458057378575586,
    min_split_gain=0.020080444638464076, max_bin=511, cat_smooth=83.19551447348591, cat_l2=15.956212913499856,
)
CAT_OPT = dict(
    depth=5, l2_leaf_reg=0.683246012387474, random_strength=0.070592450252751,
    border_count=254, bootstrap_type="Bernoulli", subsample=0.8099566629960088,
)

pipe_oof, pipe_te = {}, {}
for name in ("xgb", "lgb", "cat"):
    oof_m = np.zeros(len(y))
    te_m = np.zeros(len(Xt_p))
    for k, (a, b) in enumerate(folds_42):
        rng = np.random.default_rng(42 + k)
        a_perm = rng.permutation(a)
        cut = int(0.9 * len(a_perm))
        Xa, ya = X_p.iloc[a_perm[:cut]], y[a_perm[:cut]]
        Xv, yv = X_p.iloc[a_perm[cut:]], y[a_perm[cut:]]
        Xb = X_p.iloc[b]
        if name == "xgb":
            m = xgb.XGBClassifier(
                n_estimators=20000, learning_rate=0.02, early_stopping_rounds=200, eval_metric="auc",
                tree_method="hist", enable_categorical=True, max_cat_to_onehot=8,
                random_state=42, n_jobs=N_JOBS, **XGB_OPT
            )
            m.fit(Xa, ya, eval_set=[(Xv, yv)], verbose=False)
            oof_m[b] = m.predict_proba(Xb)[:, 1]
            te_m += m.predict_proba(Xt_p)[:, 1] / 5
        elif name == "lgb":
            m = lgb.LGBMClassifier(
                n_estimators=20000, learning_rate=0.02, random_state=42,
                verbose=-1, n_jobs=N_JOBS, **LGB_OPT
            )
            m.fit(Xa, ya, eval_set=[(Xv, yv)], eval_metric="auc", callbacks=[lgb.early_stopping(200, verbose=False)])
            oof_m[b] = m.predict_proba(Xb)[:, 1]
            te_m += m.predict_proba(Xt_p)[:, 1] / 5
        else:
            conv = lambda D: D.assign(**{c: D[c].astype(object).map(str) for c in CAT_COLS})
            m = cb.CatBoostClassifier(
                iterations=20000, learning_rate=0.02, eval_metric="AUC", od_type="Iter", od_wait=200,
                random_seed=42, thread_count=N_JOBS, verbose=False, **CAT_OPT
            )
            m.fit(conv(Xa), ya, cat_features=CAT_COLS, eval_set=(conv(Xv), yv), use_best_model=True)
            oof_m[b] = m.predict_proba(conv(Xb))[:, 1]
            te_m += m.predict_proba(conv(Xt_p))[:, 1] / 5
    pipe_oof[name], pipe_te[name] = oof_m, te_m
    print(f"  [Model C] {name.upper():4s} OOF AUC: {roc_auc_score(y, oof_m):.5f}")

  [Model C] XGB  OOF AUC: 0.89128


  [Model C] LGB  OOF AUC: 0.89193


  [Model C] CAT  OOF AUC: 0.89116


## 6. Leak-Free ECDF Rank Blend & Fixed OOF $F_2$ Threshold

- **Decision**: Instead of ranking test rows against each other, we map each model's test score to its percentile in its own **training OOF distribution (ECDF)** so every prediction depends only on training data and the single test row.
- The final binary threshold is selected to maximize **Binary $F_2$** on the blended OOF scores.

In [7]:
def ecdf_transform(oof_ref, te_vals):
    s = np.sort(oof_ref)
    f = lambda v: np.searchsorted(s, v, side="right") / len(s)
    return f(oof_ref), f(te_vals)


o_oof, o_te = ecdf_transform(c1_oof, c1_test)
s_oof, s_te = ecdf_transform(c2_oof, c2_test)

W_GBM = {"xgb": 0.1834, "lgb": 0.4848, "cat": 0.3318}
g_pairs = [ecdf_transform(pipe_oof[m], pipe_te[m]) for m in W_GBM]
g_oof = sum(w * a for w, (a, _) in zip(W_GBM.values(), g_pairs))
g_te = sum(w * b for w, (_, b) in zip(W_GBM.values(), g_pairs))
p_oof, p_te = ecdf_transform(g_oof, g_te)

b_oof = 0.70 * o_oof + 0.20 * s_oof + 0.10 * p_oof
b_te = 0.70 * o_te + 0.20 * s_te + 0.10 * p_te

grid = np.quantile(b_oof, np.linspace(0.20, 0.70, 251))
f2_scores = [fbeta_score(y, b_oof >= t, beta=2) for t in grid]
best_idx = int(np.argmax(f2_scores))
best_thr = float(grid[best_idx])

print(f"Final Blended OOF AUC:       {roc_auc_score(y, b_oof):.5f}")
print(f"Final Blended OOF Binary F2: {f2_scores[best_idx]:.5f} (threshold={best_thr:.5f}, OOF pos rate={(b_oof >= best_thr).mean():.4f})")

pred_test = (b_te >= best_thr).astype(int)
sub_t1 = pd.DataFrame({"id": test_ids, "pred": pred_test, "score": b_te})
sub_t1.to_csv(OUT_CSV, index=False, lineterminator="\n")
print(f"Saved {OUT_CSV} | rows={len(sub_t1)} | test positive rate={pred_test.mean():.4f}")
sub_t1.head()

Final Blended OOF AUC:       0.89501
Final Blended OOF Binary F2: 0.83979 (threshold=0.37264, OOF pos rate=0.6280)
Saved task1_predictions.csv | rows=25889 | test positive rate=0.5981


,id,pred,score
0,ANS00001,1,0.788829
1,ANS00002,1,0.776942
2,ANS00003,1,0.858040
3,ANS00004,1,0.681229
4,ANS00005,0,0.320968
